## Conversor ROI -> PNG
Herramienta de conversión de segmentación .ROI a mapa binario .PNG 120x120 píxeles

In [ ]:
#pip install roifile pillow numpy


In [ ]:
import os
import zipfile
from roifile import ImagejRoi
from PIL import Image, ImageDraw
import tempfile
import shutil


# CONFIGURACIÓN

folder = "dataset/CASE_1"

MRI_WIDTH = 120
MRI_HEIGHT = 120

ROI_WIDTH = 120
ROI_HEIGHT = 120

zip_names = ["cerebro.zip", "isquemia.zip"]


# PROCESO

for zip_name in zip_names:
    zip_path = os.path.join(folder, zip_name)
    if not os.path.exists(zip_path):
        print(f"ZIP no encontrado: {zip_path}")
        continue

    # Carpeta de salida
    zip_base = os.path.splitext(zip_name)[0]
    output_dir = os.path.join(folder, zip_base)
    os.makedirs(output_dir, exist_ok=True)

    print(f"Procesando {zip_path} → {output_dir}")

    # Carpeta temporal para extraer los ROI
    with tempfile.TemporaryDirectory() as tmpdir:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(tmpdir)

            for roi_name in zip_ref.namelist():
                if not roi_name.lower().endswith(".roi"):
                    continue

                roi_path = os.path.join(tmpdir, roi_name)
                roi = ImagejRoi.fromfile(roi_path)
                coords = roi.coordinates()

                # Reescalar a resolución MRI
                polygon_xy = [
                    (float(x) * MRI_WIDTH / ROI_WIDTH, float(y) * MRI_HEIGHT / ROI_HEIGHT)
                    for x, y in coords
                ]

                # Crear máscara
                mask = Image.new("L", (MRI_WIDTH, MRI_HEIGHT), 0)
                draw = ImageDraw.Draw(mask)
                draw.polygon(polygon_xy, outline=255, fill=255)

                # Guardar PNG
                out_name = os.path.basename(roi_name).replace(".roi", ".png")
                out_path = os.path.join(output_dir, out_name)
                mask.save(out_path)

print("¡Procesamiento completado para los 2 ZIPs!")
